# **STAGE 4 - MACHINE LEARNING**

## **Objectives**

* In this notebook, I am going to build and compare machine learning models capable of predicting whether a credit card customer is likely to be attrited.

## **Inputs**

* The input needed for this notebook is the cleaned dataset created in **Stage 2 - ETL**. 
* The relative path for this .csv file is: `datasets/cleaned-data/bank-churners-cleaned.csv`.
* In the notebook, I will outline which variables will be used as features (`X`) for the machine learning models.

## **Outputs**

* The output of this notebook will be two machine learning models fitted and evaluated on their effectiveness in predicting the target variable `Attrition_Flag`.

> *Ethical Considerations*: 
<br><br>*1. How the model is used*
<br><br>First and foremost, the modelling for this section has been undertaken for **educational purposes only** and **should not be used for real-life banking predictions or on real customers**.
<br><br>Whilst predicting churn may not necessarily be problematic, *how* the prediction is used matters. For example, offering a customer a helpful retention benefit is very different from reducing their services, increasing fees, or restricting access because a model predicts they are likely to leave.
<br><br>Customers predicted to churn may include people experiencing financial hardship. Targeting vulnerable customers with aggressive marketing, higher-cost products, or incentives that encourage additional borrowing can cause harm.
<br><br>Ultimately, any churn predictor that is created must be used first and foremost as a means of **improving customer experience** and not to exploit the customer's predicted behaviour and to target them with aggressive marketing.
<br><br>*2. Proxy discrimination*
<br><br>Proxy discrimination is a well-documented issue within banking machine learning. Even if you exclude protected characteristics from a model, there might be correlations that allow a model to essentially develop biases without explicitly being trained with these characteristics as features.
<br><br>

---

## **Change Working Directory**

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/credit-card-customer-churn-analysis/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/credit-card-customer-churn-analysis'

---

## **Import Packages and Libraries**

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# Import Scikit-learn Pipeline
from sklearn.pipeline import Pipeline

# Import ColumnTransformer, which is used to apply different preprocessing steps to different columns of the dataset
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder, which is used to convert categorical variables into binary vectors (one-hot encoding)
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler, which is used to standardise numerical data to make sure the data points have a balanced scale
from sklearn.preprocessing import StandardScaler

# Import SelectFromModel, which is used for feature selection based on the importance of features determined by a model
from sklearn.feature_selection import SelectFromModel

# Import ML Algorithm Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Import regression metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

---

## **Load Dataset**

Firstly, I will load the cleaned dataset `bank-churners-cleaned.csv`

In [5]:
# Load the cleaned dataset
df_ml = pd.read_csv("datasets/cleaned-data/bank-churners-cleaned.csv")

# Display the first few rows of the dataset to make sure the data has loaded correctly
df_ml.head()

,anonymised_clientnum,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Attrition_Flag_Binary
0,022e52f0a251431f954db620fd0c87ac3c523c60cc5980...,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,3,12691.0,777.0,11914.0,1.335,1144.0,42,1.625,0.061,0
1,2731de4ed9ecb2a3ab828448bfda6137e5c7571e1f7576...,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,2,8256.0,864.0,7392.0,1.541,1291.0,33,3.714,0.105,0
2,75dac624c2bbdb15b16abc0350820bcf598c012a76b102...,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,0,3418.0,0.0,3418.0,2.594,1887.0,20,2.333,0.000,0
3,1aaada0cd1f7dc23d83a171beec9f398cfac1471841843...,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,1,3313.0,2517.0,796.0,1.405,1171.0,20,2.333,0.760,0
4,1811f55b3210153f77e41ceceb61cc181d2edc4e65947e...,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,0,4716.0,0.0,4716.0,2.175,816.0,28,2.500,0.000,0


In [6]:
# Display all the columns in my DataFrame to help with feature selection
df_ml.columns

Index(['anonymised_clientnum', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
       'Attrition_Flag_Binary'],
      dtype='object')

---

## **1. Machine Learning Algorithm Selection**

The first thing I am going to do is decide what machine learning algorithms are going to be appropriate to predict the target.

| **What are you predicting?** | **Target Variable?** | **Algorithm Type** | **Examples** |
| ----- | ----- | ----- | ----- |
| Continuous Number | Yes | Regression Algorithm | Linear Regression, Decision Tree Regression, Random Forest Regression|
| Category | Yes | Classification Algorithm | Logistic Regression, Decision Tree Classification, Random Forest Classification |
| Category | No | Clustering Algorithm | K-Means Clustering |

Our target `Attrition_Flag_Binary` is a categorical target and we will therefore need a **Classification Algorithm**.

![Algorithm Selection Flow Chart](../images/algorithm-selection-classification.png)

For my prediction model, I am going to be testing the performance of **Logistic Regression** and **Random Forest Classification**.